# Lab 4 — Summarizing the old part with LangChain (EP07)

In Lab 2 we wrote `summarize_old` ourselves: once over budget → summarize the old messages, keep the recent ones word for word (Equation 18.4)
This lab does the same with LangChain's **`SummarizationMiddleware`** — the middleware we met in EP06 Lab 4

| What we wrote in Lab 2 | In `SummarizationMiddleware` |
|---|---|
| `BUDGET` + checking if we're over | `trigger=("tokens", 600)` |
| `keep_last=6` | `keep=("messages", 6)` |
| `SUMMARY_PROMPT` | `summary_prompt=` |
| `message_tokens` (Lab 1) | `token_counter=` |
| the `HISTORY` list | the agent's state in a checkpointer (thread) |

**What the framework adds:** it checks the budget itself **before every model call** in the agent loop — we never call the summarizer ourselves

The conversation is Lab 2's (**simulated data**). Every model answer and summary comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §18.2.3 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **2 places marked `✏️ TODO`** (replace `___` or `...` with your code) — each one gives the time in the video where that code is explained (▶ m:ss). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep07-lab4-langchain-summarization.en.ipynb`** (the same code as in the video).

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs under one US cent) |
| Packages | `langchain`, `langgraph`, `langchain-openai`, `langchain-core` (pinned to the same versions as EP06 Lab 4), `tokenizers`, `python-dotenv` |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

How to set up the key, and why we chose this model, is in Lab 1

In [ ]:
%pip install -q "langchain==1.4.7" "langgraph==1.2.16" "langchain-openai==1.7.2" "langchain-core==1.7.0" "tokenizers==0.23.3" python-dotenv

## 1. The model, the token counter and the conversation (from Labs 1–2)

`ChatOpenAI` pointing to OpenRouter (Gemma 4 26B pinned to DeepInfra), as in EP06 Lab 4
`USER_TURNS` are the user's 15 messages from Lab 2 — this time the agent writes every reply itself

In [ ]:
import os, re
os.environ["HF_HUB_VERBOSITY"] = "error"
from tokenizers import Tokenizer
from langchain_openai import ChatOpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

MODEL = "google/gemma-4-26b-a4b-it"
llm = ChatOpenAI(model=MODEL, base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_tokens=400,
                 extra_body={"reasoning": {"enabled": False},
                             "provider": {"order": ["deepinfra/fp8"], "allow_fallbacks": False}})
tok = Tokenizer.from_pretrained(MODEL)

PAPERS = {"P01": 2021, "P02": 2021, "P03": 2022, "P04": 2023, "P05": 2023,
          "P06": 2024, "P07": 2024, "P08": 2024, "P09": 2019, "P10": 2024}
USER_TURNS = [
    "Task for this whole chat: I'm writing a literature review on transformers for time series. I'll paste paper notes. Always answer in Thai.",
    "Correction to the task: my review only covers papers published in 2023 or later. Leave older ones out.",
    "Paper note P04: Are Transformers Effective for Time Series Forecasting? (2023). A one-layer linear model beats several transformers.",
    "Paper note P05: PatchTST (2023). Cuts each series into patches of 16 points and treats channels independently.",
    "Paper note P06: iTransformer (2024). Inverts the axes: each variable becomes a token, attention runs across variables.",
    "Thanks!",
    "Paper note P01: Informer (2021). ProbSparse self-attention cuts the cost of long inputs; tested on electricity and weather data.",
    "Paper note P07: TimesFM (2024). A decoder-only foundation model pretrained on 100 billion time points; good zero-shot forecasts.",
    "Paper note P09: Temporal Fusion Transformer (2019). LSTM encoders with attention and variable selection for interpretable forecasts.",
    "Nice, coffee break, back in five minutes.",
    "Paper note P08: Chronos (2024). Scales and quantizes values into tokens and trains a T5 language model on them.",
    "Paper note P02: Autoformer (2021). Replaces attention with auto-correlation and splits each series into trend and seasonal parts.",
    "Paper note P10: Moirai (2024). A universal forecasting transformer trained on 27 billion observations across nine domains.",
    "Haha, my cat just walked on the keyboard.",
    "Paper note P03: FEDformer (2022). Works in the frequency domain with Fourier and wavelet blocks; linear cost in sequence length.",
]
ASK = "List every paper that belongs in my review, one line each: id, title, year."

In [ ]:
ROLE = {"human": "user", "ai": "model", "system": "system", "tool": "tool"}

def gemma_counter(messages):
    return sum(len(tok.encode(f"<|turn>{ROLE[m.type]}\n{str(m.content).strip()}<turn|>\n", add_special_tokens=False).ids)
               for m in messages)

## 2. An agent + `SummarizationMiddleware`

- `trigger=("tokens", 600)` — summarize once the history goes over 600 tokens
- `keep=("messages", 6)` — keep the 6 most recent messages word for word
- `checkpointer=InMemorySaver()` — remembers each `thread_id`'s conversation across calls (EP05 Lab 5)

The first run uses LangChain's **default summary prompt**

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def make_agent(**prompt):
    summarize = SummarizationMiddleware(model=llm, trigger=("tokens", 600), keep=("messages", 6),
                                        token_counter=gemma_counter, **prompt)
    return create_agent(llm, tools=[], middleware=[summarize], checkpointer=InMemorySaver(),
                        system_prompt="You are the Cogito Learn research assistant. When the user pastes a paper note, reply with one short sentence.")

# ✏️ TODO 1 (▶ video 2:42): send one user message into the same thread and take the state's messages

def chat_all(agent, thread):
    cfg = {"configurable": {"thread_id": thread}}
    last, summaries = None, 0
    for text in USER_TURNS:
        msgs = ___
        summary = next((m.content for m in msgs if m.additional_kwargs.get("lc_source") == "summarization"), None)
        new = summary is not None and summary != last
        if new:
            summaries, last = summaries + 1, summary
        print(f"{text[:32]:32} → {len(msgs):2} messages, {gemma_counter(msgs):3} tokens {'📝 summarized' if new else ''}")

    answer = agent.invoke({"messages": [{"role": "user", "content": ASK}]}, cfg)["messages"][-1].content
    ids = sorted(set(re.findall(r"^\W*(P\d\d)", answer, re.M)))
    print(f"\nsummaries written: {summaries} · last summary: {len(tok.encode(last, add_special_tokens=False).ids)} tokens")
    print(answer, "\n→ older than 2023:", [p for p in ids if PAPERS[p] < 2023], "· right papers:", f"{sum(PAPERS[p] >= 2023 for p in ids)}/6")
    return last

In [ ]:
default_summary = chat_all(make_agent(), "default-prompt")

In [ ]:
print(default_summary[-500:])

## 3. Our prompt from Lab 2

`summary_prompt` must contain `{messages}` — the middleware replaces it with the old messages to summarize

In [ ]:
# ✏️ TODO 2 (▶ video 4:32): pass our prompt to make_agent as summary_prompt
SUMMARY_PROMPT = ("Summarize this conversation for the assistant's own memory, in English, in at most 120 words. "
                  "Keep every instruction and correction the user gave, and every paper id with its title and year. "
                  "Copy numbers exactly; add nothing.\n\n{messages}")
our_summary = chat_all(make_agent(___), "our-prompt")

In [ ]:
print(our_summary)

## Summary

| | Default prompt | Our prompt (Lab 2) |
|---|---|---|
| The summary | Sections, long enough to hit the 400-token cap (cut mid-sentence) | Under 200 tokens |
| Summaries in 15 turns | Almost every turn after the first time over budget | 1–2 |
| The final answer | 6/6 too — but with many extra model calls to summarize | 6/6, no older paper |

- The framework does the repetitive work: check the budget before every model call, split old from recent, replace them with a summary in the state
- But **its defaults weren't designed for our task** — you need to know what happens inside (Equation 18.4) to configure it well
- Further: `summarizer=` lets another (smaller, cheaper) model write the summaries, as the textbook suggests

**Exercises**
1. Remove `token_counter=gemma_counter` (use the approximate counter) and run again — at which turn is the first summary, and is anything different?
2. Set `keep=("tokens", 250)` instead of a message count — how many messages are kept?

**Next — Recap:** the whole of EP07 in one video